# Release benchmark: single-coarsener scaling

This notebook measures the public `fit`, `transform`, and full `decode` calls for
all three schema-1 coarseners over increasing tree sizes. It uses two controlled
families:

- **near-binary:** a breadth-first binary tree;
- **near-star:** one high-degree root with short repeated branches.

Every timed row must first pass an exact raw round trip and an input-nonmutation
check. Graph generation and correctness signatures are outside the timed
regions. `fit_transform` is not timed separately because it is the same public
`fit` + `transform` path and would duplicate the primary measurements.


## 0. Imports and repository checkout

Run this notebook from anywhere inside the repository. The environment record
and Git commit are saved with the results.


In [ ]:
from __future__ import annotations

import gc
import json
import os
import random
import sys
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import pandas as pd


ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").is_file() or not (ROOT / "tree_coarsening").is_dir():
    raise RuntimeError("Run this notebook from the repository root, or edit ROOT explicitly.")
BENCHMARK_DIR = ROOT / "benchmarks"

for entry in (str(ROOT), str(BENCHMARK_DIR)):
    if entry not in sys.path:
        sys.path.insert(0, entry)

from benchmark_common import (  # noqa: E402
    environment_metadata,
    make_benchmark_tree,
    make_coarsener,
    method_display_name,
    numba_is_available,
    raw_signature,
    warm_numba_backend,
)

ENVIRONMENT = environment_metadata(ROOT)
ENVIRONMENT

## 1. Controls

The default `standard` profile is intended for a visual pre-release check.
Use `smoke` to verify notebook execution and `extended` for a quiet benchmark
machine. Numba is explicitly warmed before measured rows; its cold-start time
is reported separately.


In [ ]:
PROFILES = {
    "smoke": {"sizes": (63, 127), "repeats": 1},
    "standard": {"sizes": (250, 500, 1_000, 2_000, 4_000, 8_000), "repeats": 3},
    "extended": {
        "sizes": (500, 1_000, 2_000, 4_000, 8_000, 16_000, 32_000, 64_000),
        "repeats": 5,
    },
}

PROFILE = os.environ.get("TREE_COARSENING_BENCH_PROFILE", "standard")
if PROFILE not in PROFILES:
    raise ValueError(f"Unknown benchmark profile {PROFILE!r}; choose from {tuple(PROFILES)}")

SIZES = PROFILES[PROFILE]["sizes"]
REPEATS = PROFILES[PROFILE]["repeats"]
SHAPES = ("near_binary", "near_star")
METHODS = ["star", "bpe_python", "bpe_numba", "named"]
if not numba_is_available():
    METHODS.remove("bpe_numba")

VALIDATE = "full"
BPE_NUM_MERGES = 12
BPE_MIN_PAIR_COUNT = 2
SAVE_RESULTS = True
TASK_ORDER_SEED = 20260620

{
    "profile": PROFILE,
    "sizes": SIZES,
    "repeats": REPEATS,
    "shapes": SHAPES,
    "methods": METHODS,
    "validate": VALIDATE,
    "bpe_num_merges": BPE_NUM_MERGES,
}

## 2. Build the deterministic corpora

Generation time is recorded but never folded into coarsener time. The same raw
graph object is supplied to every method; nonmutation is checked after every
row.


In [ ]:
corpora = {}
generation_rows = []
raw_signatures = {}

for shape in SHAPES:
    for n_nodes in SIZES:
        started = perf_counter()
        graph = make_benchmark_tree(shape, n_nodes, seed=TASK_ORDER_SEED + n_nodes)
        generation_seconds = perf_counter() - started
        key = (shape, n_nodes)
        corpora[key] = graph
        raw_signatures[key] = raw_signature(graph)
        generation_rows.append(
            {
                "shape": shape,
                "n_nodes": graph.number_of_nodes(),
                "n_edges": graph.number_of_edges(),
                "generation_seconds": generation_seconds,
                "root_out_degree": max(dict(graph.out_degree()).values()),
            }
        )

generation_df = pd.DataFrame(generation_rows)
generation_df

## 3. Warm-up outside measured rows

This removes import/cache effects for all methods and, most importantly,
separates Numba compilation from warmed BPE fitting.


In [ ]:
warmup_graph = make_benchmark_tree("near_binary", 127, seed=1)
warmup_rows = []

for method in [item for item in METHODS if item != "bpe_numba"]:
    model = make_coarsener(
        method,
        model_id=f"single-warmup-{method}",
        bpe_num_merges=2,
        bpe_min_pair_count=1,
    )
    started = perf_counter()
    model.fit([warmup_graph], validate=False)
    encoded = model.transform(warmup_graph, validate=False)
    decoded = model.decode(encoded, validate=False)
    elapsed = perf_counter() - started
    assert raw_signature(decoded) == raw_signature(warmup_graph)
    warmup_rows.append({"method": method, "warmup_seconds": elapsed})

numba_cold_seconds = warm_numba_backend(validate=False) if "bpe_numba" in METHODS else None
if numba_cold_seconds is not None:
    warmup_rows.append({"method": "bpe_numba", "warmup_seconds": numba_cold_seconds})

pd.DataFrame(warmup_rows).assign(method=lambda frame: frame["method"].map(method_display_name))

## 4. Timed sweep

Each repetition constructs and fits a fresh coarsener. Task order is shuffled
deterministically to reduce systematic ordering bias. The timed total is the
sum of the three public calls, excluding model construction, graph generation,
and correctness signatures.


In [ ]:
tasks = [
    (repeat, shape, n_nodes, method)
    for repeat in range(REPEATS)
    for shape in SHAPES
    for n_nodes in SIZES
    for method in METHODS
]
random.Random(TASK_ORDER_SEED).shuffle(tasks)

rows = []
for task_index, (repeat, shape, n_nodes, method) in enumerate(tasks, start=1):
    graph = corpora[(shape, n_nodes)]
    expected_raw = raw_signatures[(shape, n_nodes)]
    model = make_coarsener(
        method,
        model_id=f"single-{method}-{shape}-{n_nodes}-{repeat}",
        bpe_num_merges=BPE_NUM_MERGES,
        bpe_min_pair_count=BPE_MIN_PAIR_COUNT,
    )

    started = perf_counter()
    model.fit([graph], validate=VALIDATE)
    fit_seconds = perf_counter() - started

    started = perf_counter()
    encoded = model.transform(graph, validate=VALIDATE)
    transform_seconds = perf_counter() - started

    started = perf_counter()
    decoded = model.decode(encoded, validate=VALIDATE)
    decode_seconds = perf_counter() - started

    roundtrip_ok = raw_signature(decoded) == expected_raw
    input_unchanged = raw_signature(graph) == expected_raw
    if not roundtrip_ok or not input_unchanged:
        raise AssertionError(
            f"Correctness failure for method={method}, shape={shape}, n={n_nodes}, "
            f"repeat={repeat}: roundtrip={roundtrip_ok}, unchanged={input_unchanged}"
        )

    rule_count = len(model.encoder_.rules)
    rows.append(
        {
            "repeat": repeat,
            "shape": shape,
            "n_nodes": n_nodes,
            "method": method,
            "method_name": method_display_name(method),
            "fit_seconds": fit_seconds,
            "transform_seconds": transform_seconds,
            "decode_seconds": decode_seconds,
            "total_seconds": fit_seconds + transform_seconds + decode_seconds,
            "encoded_nodes": encoded.number_of_nodes(),
            "compression_ratio": encoded.number_of_nodes() / n_nodes,
            "rule_count": rule_count,
            "backend_used": getattr(model, "backend_used_", None),
            "roundtrip_ok": roundtrip_ok,
            "input_unchanged": input_unchanged,
        }
    )
    if task_index % max(1, len(tasks) // 10) == 0:
        print(f"completed {task_index}/{len(tasks)} rows")
    del model, encoded, decoded
    gc.collect()

results_df = pd.DataFrame(rows).sort_values(["shape", "method", "n_nodes", "repeat"])
results_df.head()

## 5. Aggregate and verify

The median is the primary estimate. Minimum and maximum values remain in the
summary so noisy points are visible rather than silently discarded.


In [ ]:
if not results_df["roundtrip_ok"].all() or not results_df["input_unchanged"].all():
    raise AssertionError("At least one benchmark row failed correctness checks.")

summary_df = (
    results_df.groupby(["shape", "n_nodes", "method", "method_name"], as_index=False)
    .agg(
        fit_median=("fit_seconds", "median"),
        transform_median=("transform_seconds", "median"),
        decode_median=("decode_seconds", "median"),
        total_median=("total_seconds", "median"),
        total_min=("total_seconds", "min"),
        total_max=("total_seconds", "max"),
        encoded_nodes=("encoded_nodes", "median"),
        compression_ratio=("compression_ratio", "median"),
        rule_count=("rule_count", "median"),
    )
    .sort_values(["shape", "method", "n_nodes"])
)

summary_df["nodes_per_total_second"] = summary_df["n_nodes"] / summary_df["total_median"]
summary_df

## 6. Total public-call time by shape

Look for smooth scaling, stable method ordering, and abrupt slope changes. The
spread table above is as important as the median line when a machine is noisy.


In [ ]:
for shape in SHAPES:
    figure = plt.figure(figsize=(10, 6))
    axis = figure.gca()
    subset = summary_df[summary_df["shape"] == shape]
    for method_name, group in subset.groupby("method_name", sort=False):
        group = group.sort_values("n_nodes")
        axis.plot(group["n_nodes"], group["total_median"], marker="o", label=method_name)
    axis.set_xscale("log", base=2)
    axis.set_yscale("log")
    axis.set_xlabel("Raw nodes")
    axis.set_ylabel("Median fit + transform + decode seconds")
    axis.set_title(f"Single-coarsener end-to-end scaling: {shape}")
    axis.grid(True, which="both", alpha=0.25)
    axis.legend()
    plt.show()

## 7. Phase timing and compression

A total-time regression can come from validation/fitting, concrete rewiring, or
exact decoding. Compression ratio and rule count guard against mistaking a
no-op for a speedup.


In [ ]:
for phase, ylabel in (
    ("fit_median", "Median fit seconds"),
    ("transform_median", "Median transform seconds"),
    ("decode_median", "Median decode seconds"),
):
    for shape in SHAPES:
        figure = plt.figure(figsize=(10, 6))
        axis = figure.gca()
        subset = summary_df[summary_df["shape"] == shape]
        for method_name, group in subset.groupby("method_name", sort=False):
            group = group.sort_values("n_nodes")
            axis.plot(group["n_nodes"], group[phase], marker="o", label=method_name)
        axis.set_xscale("log", base=2)
        axis.set_yscale("log")
        axis.set_xlabel("Raw nodes")
        axis.set_ylabel(ylabel)
        axis.set_title(f"{phase.replace('_median', '').title()} scaling: {shape}")
        axis.grid(True, which="both", alpha=0.25)
        axis.legend()
        plt.show()

for shape in SHAPES:
    figure = plt.figure(figsize=(10, 6))
    axis = figure.gca()
    subset = summary_df[summary_df["shape"] == shape]
    for method_name, group in subset.groupby("method_name", sort=False):
        group = group.sort_values("n_nodes")
        axis.plot(group["n_nodes"], group["compression_ratio"], marker="o", label=method_name)
    axis.set_xscale("log", base=2)
    axis.set_xlabel("Raw nodes")
    axis.set_ylabel("Encoded nodes / raw nodes")
    axis.set_title(f"Compression ratio: {shape}")
    axis.grid(True, which="both", alpha=0.25)
    axis.legend()
    plt.show()

## 8. Save reproducible results

The JSON file records benchmark controls and the execution environment. Keep
both files when comparing commits.


In [ ]:
if SAVE_RESULTS:
    output_dir = ROOT / "benchmark_results"
    output_dir.mkdir(exist_ok=True)
    csv_path = output_dir / f"coarsener_scaling_{PROFILE}.csv"
    json_path = output_dir / f"coarsener_scaling_{PROFILE}_metadata.json"
    results_df.to_csv(csv_path, index=False)
    metadata = {
        "environment": ENVIRONMENT,
        "profile": PROFILE,
        "sizes": list(SIZES),
        "repeats": REPEATS,
        "shapes": list(SHAPES),
        "methods": list(METHODS),
        "validate": VALIDATE,
        "bpe_num_merges": BPE_NUM_MERGES,
        "bpe_min_pair_count": BPE_MIN_PAIR_COUNT,
        "numba_cold_warmup_seconds": numba_cold_seconds,
    }
    json_path.write_text(json.dumps(metadata, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    print(csv_path.relative_to(ROOT))
    print(json_path.relative_to(ROOT))

## 9. Release interpretation checklist

- Confirm every correctness column is `True`.
- Confirm rule counts and compression ratios remain nontrivial across sizes.
- Compare only like-for-like environments and validation levels.
- Treat the Numba warm-up value as cold-start cost; compare warmed rows to warmed
  rows.
- Investigate sustained slope changes or a repeatable 10–15%+ regression, not a
  single noisy point.
- Repeat the same public-call timings on representative project trees before
  release; these controlled shapes are necessary but not sufficient.
